# Lesson 5.1: Exposing the Agent via A2A

In the main Lesson 5.0 we built an agent that *consumes* tools via MCP. This notebook shows the other half: how that same agent can be *consumed* by **other agents** via the **A2A protocol**.

## 1. Why A2A?

Two protocols, two different jobs:

| Protocol | Direction | Purpose |
|---|---|---|
| **MCP** | agent ↔ tools | Give one agent access to a set of tools |
| **A2A** | agent ↔ agent | Let agents discover and delegate to other agents |

Real systems will have several specialist agents. A scheduling agent might delegate "which technicians need retraining?" to our maintenance analyst rather than reimplement the analysis. A2A is the standard for that handoff.

Every A2A agent publishes two things:

1. **An Agent Card** at `/.well-known/agent.json` — a JSON document describing the agent and the skills it offers.
2. **A JSON-RPC endpoint** that accepts skill calls and returns messages.

If you've seen OAuth's discovery doc or OpenAPI's `/openapi.json`, the pattern will feel familiar — declare capabilities at a well-known URL, let consumers introspect.

**A2A is a spec, not a library.** We can implement it directly on top of FastAPI in a few cells. There are SDKs available (Google's `a2a-sdk` for example), but the protocol itself is small enough that seeing it raw is the better starting point.

## 2. Setup

The Lesson 4 MCP server should still be running on port 8001, and `OPENAI_API_KEY` should be in `.env`.

In [ ]:
import os
import json
import socket
import threading
import time
import uuid
from pathlib import Path
from dotenv import load_dotenv

_env = Path().resolve().parent / '.env'
if not _env.exists():
    _env = Path().resolve() / '.env'
load_dotenv(_env)

assert os.environ.get('OPENAI_API_KEY'), 'OPENAI_API_KEY not found in .env'

MCP_URL = 'http://localhost:8001/sse'
AGENT_PORT = 8002
AGENT_URL = f'http://localhost:{AGENT_PORT}'

_check = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
_check.settimeout(1.0)
assert _check.connect_ex(('localhost', 8001)) == 0, (
    'MCP server not reachable on :8001. Run mcp_tools/lesson_04_mcp_tools.ipynb first.'
)
_check.close()
print('Environment ready ✓')

## 3. Declare the Agent Card

The Agent Card is the contract. Other agents read it before they call anything — same way you'd read an OpenAPI spec before integrating with a REST API.

Each **skill** describes one thing the agent can do, in natural language plus a structured `id`. Skills are **discoverable** — peer agents pick them based on the description and the examples, not by reading our code.

Below we build the card as a plain dict matching the A2A spec. This is exactly the JSON that will be served from `/.well-known/agent.json`.

In [ ]:
AGENT_CARD = {
    'name': 'Maintenance Analyst',
    'description': 'Investigates machine usage and recommends retraining based on factory data.',
    'url': AGENT_URL,
    'version': '0.1.0',
    'capabilities': {
        'streaming': False,
        'pushNotifications': False,
    },
    'defaultInputModes': ['text/plain'],
    'defaultOutputModes': ['text/plain'],
    'skills': [
        {
            'id': 'analyze-training-needs',
            'name': 'Analyze Training Needs',
            'description': (
                'Identify technicians whose maintenance patterns suggest they would benefit '
                'from retraining on specific failure types. Returns ranked recommendations '
                'with the evidence behind each one.'
            ),
            'tags': ['maintenance', 'training', 'analysis'],
            'examples': [
                'Which technicians need retraining based on the last 90 days?',
                'Find the top training gaps for hydraulic issues.',
            ],
        },
        {
            'id': 'find-top-issues',
            'name': 'Find Top Maintenance Issues',
            'description': (
                'Surface the (machine, failure type) pairs responsible for the most '
                'downtime in a given period.'
            ),
            'tags': ['maintenance', 'analysis'],
            'examples': ['What are the top maintenance issues this month?'],
        },
    ],
}

print(json.dumps(AGENT_CARD, indent=2))

That JSON is exactly what peer agents fetch from `/.well-known/agent.json`. A few things worth noticing:

- The `description` and `examples` on each skill are the **routing signal**. Just like with MCP tool descriptions in Lesson 4, what we write here determines whether peer agents pick us for the right kinds of requests.
- `capabilities` declares what *transport features* the agent supports (streaming, push). Peers see this before they try to subscribe.
- The `url` is where the JSON-RPC endpoint lives. Discovery is at `<url>/.well-known/agent.json`; invocation is at `<url>`.

## 4. Wire skills to the Lesson 5 agent

When a peer agent calls one of our skills, we want the same Lesson 5 agent to handle it. The `execute_skill` function below is the bridge: take the user's text, spin up the agent against the MCP server, return the final answer.

In [ ]:
from agents import Agent, Runner
from agents.mcp import MCPServerSse, MCPServerSseParams

INSTRUCTIONS = (
    'You are a manufacturing maintenance analyst. '
    'Use the available tools to investigate the data and answer the request. '
    'Cite IDs and metrics; do not invent details.'
)

async def execute_skill(user_text: str) -> str:
    """Run the Lesson 5 agent against a single user message and return its final answer."""
    async with MCPServerSse(params=MCPServerSseParams(url=MCP_URL)) as mcp:
        agent = Agent(
            name='maintenance_analyst',
            model='gpt-4o-mini',
            instructions=INSTRUCTIONS,
            mcp_servers=[mcp],
        )
        result = await Runner.run(agent, user_text)
    return result.final_output

print('Executor defined ✓')

## 5. Stand up the A2A server

Two endpoints, that's the whole protocol:

- `GET /.well-known/agent.json` — return the Agent Card.
- `POST /` — handle JSON-RPC 2.0 requests. We only implement `message/send` (the one peers actually need to invoke a skill).

We run it on port 8002 in a background thread so the notebook stays interactive.

In [ ]:
import uvicorn
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse

app = FastAPI()

@app.get('/.well-known/agent.json')
def discover():
    return AGENT_CARD


def _jsonrpc_error(req_id, code: int, message: str):
    return JSONResponse({
        'jsonrpc': '2.0',
        'id': req_id,
        'error': {'code': code, 'message': message},
    })


def _extract_user_text(message: dict) -> str:
    """Pull text out of an A2A message's parts list."""
    parts = message.get('parts', [])
    chunks = [p.get('text', '') for p in parts if p.get('kind') == 'text']
    return '\n'.join(c for c in chunks if c)


@app.post('/')
async def jsonrpc(request: Request):
    payload = await request.json()
    req_id = payload.get('id')
    method = payload.get('method')

    if method != 'message/send':
        return _jsonrpc_error(req_id, -32601, f'Method not supported: {method}')

    params = payload.get('params') or {}
    message = params.get('message') or {}
    user_text = _extract_user_text(message)
    if not user_text:
        return _jsonrpc_error(req_id, -32602, 'No text content in message.parts')

    answer = await execute_skill(user_text)

    # A2A response shape: a completed Task with an agent message in its history.
    task = {
        'id': str(uuid.uuid4()),
        'status': {'state': 'completed'},
        'history': [
            message,
            {
                'role': 'agent',
                'messageId': str(uuid.uuid4()),
                'parts': [{'kind': 'text', 'text': answer}],
            },
        ],
    }
    return JSONResponse({'jsonrpc': '2.0', 'id': req_id, 'result': task})


def _port_open(port: int) -> bool:
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.settimeout(0.5)
    ok = s.connect_ex(('localhost', port)) == 0
    s.close()
    return ok

if _port_open(AGENT_PORT):
    print(f'A2A server already running on port {AGENT_PORT} ✓')
else:
    def _serve():
        uvicorn.run(app, host='0.0.0.0', port=AGENT_PORT, log_level='warning')
    threading.Thread(target=_serve, daemon=True).start()
    for _ in range(20):
        if _port_open(AGENT_PORT):
            print(f'A2A server up on {AGENT_URL} ✓')
            break
        time.sleep(0.5)
    else:
        print('A2A server did not come up in time.')

## 6. Call the agent over A2A

Two things any peer agent does:

1. **Discovery** — GET `/.well-known/agent.json` to learn what skills are offered.
2. **Invocation** — POST a JSON-RPC `message/send` request with the user message.

We do both directly with `httpx` so you can see the wire format.

In [ ]:
import httpx

resp = httpx.get(f'{AGENT_URL}/.well-known/agent.json', timeout=5.0)
resp.raise_for_status()
card = resp.json()

print(f"Discovered agent: {card['name']} (v{card['version']})")
print('Skills offered:')
for s in card['skills']:
    print(f"  • {s['id']:28s}  {s['description'][:70]}")

In [ ]:
rpc_request = {
    'jsonrpc': '2.0',
    'id': str(uuid.uuid4()),
    'method': 'message/send',
    'params': {
        'message': {
            'role': 'user',
            'messageId': str(uuid.uuid4()),
            'parts': [
                {'kind': 'text', 'text': 'What are the top 3 maintenance issues right now?'}
            ],
        }
    },
}

resp = httpx.post(f'{AGENT_URL}/', json=rpc_request, timeout=120.0)
resp.raise_for_status()
envelope = resp.json()

# Pull the agent's reply out of the task history.
task = envelope['result']
for msg in task['history']:
    if msg['role'] == 'agent':
        for part in msg['parts']:
            if part.get('kind') == 'text':
                print(part['text'])
                break

## 7. Summary

| Section | What we did |
|---|---|
| Why A2A | MCP is agent↔tools; A2A is agent↔agent. Two protocols, two jobs. |
| Agent Card | Declared the card as a JSON dict — the menu peer agents read to decide whether to call us. |
| Executor | Wired skill invocations to the same Lesson 5 agent (MCP-backed). |
| Server | Two FastAPI routes implement the whole protocol surface: discovery + JSON-RPC. |
| Call | Discovered the card and invoked a skill via `message/send`. |

The big idea: an agent isn't just something *you* talk to. It's also something *other agents* talk to. The Agent Card is how they find each other; A2A is how they speak.